In [1]:
from pathlib import Path
import sys

BASE_DIR = Path().resolve().parent
sys.path.append(str(BASE_DIR))

In [2]:
bronze_path = BASE_DIR / "data" / "bronze"

In [3]:
list(bronze_path.glob("*.parquet"))

[]

In [4]:
from src.utils import get_spark
spark = get_spark()

In [5]:
df_prior = spark.read.parquet(
    str(bronze_path / "order_products__prior")
)

df_train = spark.read.parquet(
    str(bronze_path / "order_products__train")
)
df_order_items = df_prior.unionByName(df_train)

In [6]:
df_order_items.show()

+--------+----------+-----------------+---------+
|order_id|product_id|add_to_cart_order|reordered|
+--------+----------+-----------------+---------+
|       2|     33120|                1|        1|
|       2|     28985|                2|        1|
|       2|      9327|                3|        0|
|       2|     45918|                4|        1|
|       2|     30035|                5|        0|
|       2|     17794|                6|        1|
|       2|     40141|                7|        1|
|       2|      1819|                8|        1|
|       2|     43668|                9|        0|
|       3|     33754|                1|        1|
|       3|     24838|                2|        1|
|       3|     17704|                3|        1|
|       3|     21903|                4|        1|
|       3|     17668|                5|        1|
|       3|     46667|                6|        1|
|       3|     17461|                7|        1|
|       3|     32665|                8|        1|


In [7]:
df_prior.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)



In [8]:
df_train.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)



In [9]:
df_order_items.show()

+--------+----------+-----------------+---------+
|order_id|product_id|add_to_cart_order|reordered|
+--------+----------+-----------------+---------+
|       2|     33120|                1|        1|
|       2|     28985|                2|        1|
|       2|      9327|                3|        0|
|       2|     45918|                4|        1|
|       2|     30035|                5|        0|
|       2|     17794|                6|        1|
|       2|     40141|                7|        1|
|       2|      1819|                8|        1|
|       2|     43668|                9|        0|
|       3|     33754|                1|        1|
|       3|     24838|                2|        1|
|       3|     17704|                3|        1|
|       3|     21903|                4|        1|
|       3|     17668|                5|        1|
|       3|     46667|                6|        1|
|       3|     17461|                7|        1|
|       3|     32665|                8|        1|


In [10]:
df_order_items.show()

+--------+----------+-----------------+---------+
|order_id|product_id|add_to_cart_order|reordered|
+--------+----------+-----------------+---------+
|       2|     33120|                1|        1|
|       2|     28985|                2|        1|
|       2|      9327|                3|        0|
|       2|     45918|                4|        1|
|       2|     30035|                5|        0|
|       2|     17794|                6|        1|
|       2|     40141|                7|        1|
|       2|      1819|                8|        1|
|       2|     43668|                9|        0|
|       3|     33754|                1|        1|
|       3|     24838|                2|        1|
|       3|     17704|                3|        1|
|       3|     21903|                4|        1|
|       3|     17668|                5|        1|
|       3|     46667|                6|        1|
|       3|     17461|                7|        1|
|       3|     32665|                8|        1|


In [11]:
df_products = spark.read.parquet(
    str(bronze_path / "products")
)

df_fact_order_items = (
    df_order_items
    .join(df_products, on="product_id", how="left")
)
df_fact_order_items.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle_id: string (nullable = true)
 |-- department_id: string (nullable = true)



In [12]:
df_fact_order_items.show(20)

+----------+--------+-----------------+---------+--------------------+--------+-------------+
|product_id|order_id|add_to_cart_order|reordered|        product_name|aisle_id|department_id|
+----------+--------+-----------------+---------+--------------------+--------+-------------+
|     33120|       2|                1|        1|  Organic Egg Whites|      86|           16|
|     28985|       2|                2|        1|Michigan Organic ...|      83|            4|
|      9327|       2|                3|        0|       Garlic Powder|     104|           13|
|     45918|       2|                4|        1|      Coconut Butter|      19|           13|
|     30035|       2|                5|        0|   Natural Sweetener|      17|           13|
|     17794|       2|                6|        1|             Carrots|      83|            4|
|     40141|       2|                7|        1|Original Unflavor...|     105|           13|
|      1819|       2|                8|        1|All Natural

In [13]:
df_orders = spark.read.parquet(
    str(bronze_path / "orders")
)

df_fact = (
    df_fact_order_items
    .join(df_orders, on="order_id", how="left")
)

df_fact.count()

33819106

In [14]:
df_fact.show()

+--------+----------+-----------------+---------+--------------------+--------+-------------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|product_id|add_to_cart_order|reordered|        product_name|aisle_id|department_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+----------+-----------------+---------+--------------------+--------+-------------+-------+--------+------------+---------+-----------------+----------------------+
|  570852|     23085|                1|        1|Montauk Soft Bake...|      61|           19|  74739|   train|          48|        3|               17|                  16.0|
|  570852|     28199|                2|        1|    Clementines, Bag|     123|            4|  74739|   train|          48|        3|               17|                  16.0|
|  570852|     32478|                3|        1| Reduced Fat 2% Milk|      84|           16|  74739|   train|          48|  

In [15]:
df_fact.filter(df_fact.order_id.isNull()).count()

0

In [16]:
df_fact.filter(df_fact.product_id.isNull()).count()

0

In [17]:
df_orders.show()

+--------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+-------+--------+------------+---------+-----------------+----------------------+
| 2539329|      1|   prior|           1|        2|                8|                  NULL|
| 2398795|      1|   prior|           2|        3|                7|                  15.0|
|  473747|      1|   prior|           3|        3|               12|                  21.0|
| 2254736|      1|   prior|           4|        4|                7|                  29.0|
|  431534|      1|   prior|           5|        4|               15|                  28.0|
| 3367565|      1|   prior|           6|        2|                7|                  19.0|
|  550135|      1|   prior|           7|        1|                9|                  20.0|
| 3108588|      1|   prior|           8|        1|               14|            

In [18]:
df_orders.filter(df_orders.days_since_prior_order.isNull()).count()

206209

In [19]:
df_aisles = spark.read.parquet(
    str(bronze_path / "aisles")
)
df_departments = spark.read.parquet(
    str(bronze_path / "departments")
)

In [20]:
dim_product = (
    df_products
    .join(df_aisles, on="aisle_id", how="left")
    .join(df_departments, on="department_id", how="left")
)

In [21]:
dim_product.show()

+-------------+--------+----------+--------------------+--------------------+---------------+
|department_id|aisle_id|product_id|        product_name|               aisle|     department|
+-------------+--------+----------+--------------------+--------------------+---------------+
|           19|      61|         1|Chocolate Sandwic...|       cookies cakes|         snacks|
|           13|     104|         2|    All-Seasons Salt|   spices seasonings|         pantry|
|            7|      94|         3|Robust Golden Uns...|                 tea|      beverages|
|            1|      38|         4|Smart Ones Classi...|        frozen meals|         frozen|
|           13|       5|         5|Green Chile Anyti...|marinades meat pr...|         pantry|
|           11|      11|         6|        Dry Nose Oil|    cold flu allergy|  personal care|
|            7|      98|         7|Pure Coconut Wate...|       juice nectars|      beverages|
|            1|     116|         8|Cut Russet Potato...|    